# EXP_08: Peak SOTA Pipeline (Optimized for 4GB GPU VRAM)

**Objective:** Reach peak **🏆 98%+ SOTA Classification Accuracy** on CIFAR-10 test set without running out of GPU memory (CUDA OOM).

### 🛠️ Key VRAM Optimizations for 4GB GPUs:
1. **Backbone:** `ConvNeXt-Small` / `ConvNeXt-Tiny` (Perfect size for 4GB VRAM).
2. **Precision:** Automatic Mixed Precision (`AMP - FP16`) for 50% memory savings.
3. **Batch Size:** Batch = 16 with Gradient Accumulation = 2.
4. **Memory Management:** Explicit `del` and `gc.collect()` to prevent Jupyter VRAM leaks.

In [1]:
# Cell 1: Environment Setup & Aggressive GPU VRAM Cleanup
import os
import sys
import time
import math
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Aggressive CUDA VRAM Cleanup
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.data.dataloader import get_cifar10_loaders
from src.eval.evaluate_model import CIFAR10_CLASSES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[Setup] PyTorch Version: {torch.__version__}")
print(f"[Setup] Execution Device: {device}")
if torch.cuda.is_available():
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    free_mem = torch.cuda.mem_get_info()[0] / (1024**3)
    print(f"[Setup] GPU: {torch.cuda.get_device_name(0)} ({free_mem:.2f} GB Free / {total_mem:.2f} GB Total VRAM)")


[Setup] PyTorch Version: 2.13.0+cu130
[Setup] Execution Device: cuda
[Setup] GPU: NVIDIA GeForce RTX 3050 Laptop GPU (3.57 GB Free / 3.68 GB Total VRAM)


## 1. Mixup & CutMix Data Collator

In [2]:
# Cell 2: Define Mixup & CutMix Collator
class MixupCutMixCollator:
    def __init__(self, mixup_alpha=0.8, cutmix_alpha=1.0, prob=0.8, num_classes=10):
        self.mixup_alpha = mixup_alpha
        self.cutmix_alpha = cutmix_alpha
        self.prob = prob
        self.num_classes = num_classes

    def rand_bbox(self, size, lam):
        W = size[2]
        H = size[3]
        cut_rat = np.sqrt(1. - lam)
        cut_w = int(W * cut_rat)
        cut_h = int(H * cut_rat)
        cx = np.random.randint(W)
        cy = np.random.randint(H)
        bbx1 = np.clip(cx - cut_w // 2, 0, W)
        bby1 = np.clip(cy - cut_h // 2, 0, H)
        bbx2 = np.clip(cx + cut_w // 2, 0, W)
        bby2 = np.clip(cy + cut_h // 2, 0, H)
        return bbx1, bby1, bbx2, bby2

    def __call__(self, batch):
        images, labels = torch.utils.data.dataloader.default_collate(batch)
        if np.random.rand() > self.prob:
            return images, F.one_hot(labels, self.num_classes).float()

        one_hot_labels = F.one_hot(labels, self.num_classes).float()
        use_cutmix = np.random.rand() > 0.5

        if use_cutmix and self.cutmix_alpha > 0:
            lam = np.random.beta(self.cutmix_alpha, self.cutmix_alpha)
            rand_index = torch.randperm(images.size(0))
            target_a = one_hot_labels
            target_b = one_hot_labels[rand_index]
            bbx1, bby1, bbx2, bby2 = self.rand_bbox(images.size(), lam)
            images[:, :, bbx1:bbx2, bby1:bby2] = images[rand_index, :, bbx1:bbx2, bby1:bby2]
            lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (images.size(2) * images.size(3)))
            targets = target_a * lam + target_b * (1. - lam)
        else:
            lam = np.random.beta(self.mixup_alpha, self.mixup_alpha)
            rand_index = torch.randperm(images.size(0))
            images = lam * images + (1 - lam) * images[rand_index]
            targets = lam * one_hot_labels + (1 - lam) * one_hot_labels[rand_index]

        return images, targets

print("[Data] MixupCutMixCollator ready.")


[Data] MixupCutMixCollator ready.


In [3]:
# Cell 3: Data Loaders (224x224 Resolution & Batch Size 16 for 4GB VRAM)
IMAGE_SIZE = 224  # 224x224 resolution fits easily in 4GB GPU VRAM
BATCH_SIZE = 16   # Batch Size 16 avoids OOM

train_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.RandomHorizontalFlip(p=0.5),
    T.RandAugment(num_ops=2, magnitude=12),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    T.RandomErasing(p=0.25, scale=(0.02, 0.2))
])

eval_transform = T.Compose([
    T.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=T.InterpolationMode.BICUBIC),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

mixup_collator = MixupCutMixCollator(mixup_alpha=0.8, cutmix_alpha=1.0, prob=0.8)

train_loader, val_loader, test_loader = get_cifar10_loaders(
    train_transform=train_transform,
    eval_transform=eval_transform,
    batch_size=BATCH_SIZE,
    num_workers=0
)

print(f"[Data] Configured: Resolution={IMAGE_SIZE}x{IMAGE_SIZE}, Batch Size={BATCH_SIZE}")


[Data] Configured: Resolution=224x224, Batch Size=16


## 2. Model Builder & EMA

In [4]:
# Cell 4: Model Builder (ConvNeXt-Small / Tiny for 4GB VRAM)
def build_model_sota(model_name="convnext_small", num_classes=10, device=device):
    if model_name == "convnext_small":
        from torchvision.models import convnext_small, ConvNeXt_Small_Weights
        model = convnext_small(weights=ConvNeXt_Small_Weights.DEFAULT)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)
    elif model_name == "convnext_tiny":
        from torchvision.models import convnext_tiny, ConvNeXt_Tiny_Weights
        model = convnext_tiny(weights=ConvNeXt_Tiny_Weights.DEFAULT)
        in_features = model.classifier[2].in_features
        model.classifier[2] = nn.Linear(in_features, num_classes)
    return model.to(device)

class ExponentialMovingAverage:
    def __init__(self, model, decay=0.9999):
        self.model = model
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                new_average = (1.0 - self.decay) * param.data + self.decay * self.shadow[name]
                self.shadow[name] = new_average.clone()

    def apply_shadow(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                self.backup[name] = param.data
                param.data = self.shadow[name]

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad:
                param.data = self.backup[name]
        self.backup = {}

print("[Model] Builder and EMA defined.")


[Model] Builder and EMA defined.


## 3. Training Loop with Automatic Mixed Precision (AMP) & Memory Cleanup

In [5]:
# Cell 5: AMP Training & Evaluation Functions
scaler = torch.amp.GradScaler('cuda', enabled=(device.type == 'cuda'))

def train_epoch_exp08(model, loader, criterion, optimizer, ema, device, collator=None, grad_accum_steps=2):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    optimizer.zero_grad()

    for i, (images, labels) in enumerate(loader):
        if collator is not None:
            images, targets = collator(list(zip(images, labels)))
            images, targets = images.to(device), targets.to(device)
        else:
            images = images.to(device)
            targets = F.one_hot(labels, 10).float().to(device)

        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, targets) / grad_accum_steps

        scaler.scale(loss).backward()

        if (i + 1) % grad_accum_steps == 0 or (i + 1) == len(loader):
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad()
            if ema is not None:
                ema.update()

        running_loss += loss.item() * grad_accum_steps * images.size(0)
        _, predicted = outputs.max(1)
        _, target_class = targets.max(1)
        total += labels.size(0)
        correct += predicted.eq(target_class).sum().item()

    return running_loss / total, 100.0 * correct / total

@torch.inference_mode()
def evaluate_exp08(model, loader, device):
    model.eval()
    criterion = nn.CrossEntropyLoss()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        with torch.amp.autocast('cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    return running_loss / total, 100.0 * correct / total

print("[Training] AMP Enabled.")


[Training] AMP Enabled.


## 4. Run EXP-08 Fine-Tuning Execution

In [6]:
# Cell 6: Execute EXP-08 Training Pipeline
# Clear leftover CUDA memory before allocation
gc.collect()
torch.cuda.empty_cache()

epochs = 10
# Use convnext_small or convnext_tiny for 4GB GPUs
model = build_model_sota(model_name="convnext_small", num_classes=10, device=device)
ema = ExponentialMovingAverage(model, decay=0.9999)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)
criterion = nn.CrossEntropyLoss()

checkpoint_dir = os.path.join(PROJECT_ROOT, "experiments", "checkpoints")
os.makedirs(checkpoint_dir, exist_ok=True)
best_ckpt_path = os.path.join(checkpoint_dir, "exp08_convnext_small_sota_best.pt")

print(f"=== STARTING EXP-08 SOTA TRAINING ({epochs} Epochs, AMP Enabled) ===")
best_val_acc = 0.0

for epoch in range(1, epochs + 1):
    t0 = time.time()
    train_loss, train_acc = train_epoch_exp08(model, train_loader, criterion, optimizer, ema, device, collator=mixup_collator)
    scheduler.step()
    
    # Validation with EMA weights
    ema.apply_shadow()
    ema_val_loss, ema_val_acc = evaluate_exp08(model, val_loader, device)
    ema.restore()
    
    elapsed = time.time() - t0
    print(f"Epoch {epoch:2d}/{epochs:2d} [{elapsed:.1f}s] | Train Loss: {train_loss:.4f} | EMA Val Acc: {ema_val_acc:.2f}%")
    
    if ema_val_acc > best_val_acc:
        best_val_acc = ema_val_acc
        ema.apply_shadow()
        torch.save(model.state_dict(), best_ckpt_path)
        ema.restore()
        print(f"  --> New Peak EMA Val Accuracy: {best_val_acc:.2f}% Saved to {best_ckpt_path}")


Downloading: "https://download.pytorch.org/models/convnext_small-0c510722.pth" to /home/bush/.cache/torch/hub/checkpoints/convnext_small-0c510722.pth


100%|██████████| 192M/192M [04:40<00:00, 718kB/s]  


=== STARTING EXP-08 SOTA TRAINING (10 Epochs, AMP Enabled) ===
Epoch  1/10 [671.8s] | Train Loss: 0.8490 | EMA Val Acc: 54.60%
  --> New Peak EMA Val Accuracy: 54.60% Saved to /home/bush/Desktop/Deeplearning_Course_UTH/experiments/checkpoints/exp08_convnext_small_sota_best.pt
Epoch  2/10 [664.8s] | Train Loss: 0.7126 | EMA Val Acc: 90.82%
  --> New Peak EMA Val Accuracy: 90.82% Saved to /home/bush/Desktop/Deeplearning_Course_UTH/experiments/checkpoints/exp08_convnext_small_sota_best.pt
Epoch  3/10 [662.8s] | Train Loss: 0.6922 | EMA Val Acc: 96.22%
  --> New Peak EMA Val Accuracy: 96.22% Saved to /home/bush/Desktop/Deeplearning_Course_UTH/experiments/checkpoints/exp08_convnext_small_sota_best.pt
Epoch  4/10 [672.1s] | Train Loss: 0.6628 | EMA Val Acc: 97.44%
  --> New Peak EMA Val Accuracy: 97.44% Saved to /home/bush/Desktop/Deeplearning_Course_UTH/experiments/checkpoints/exp08_convnext_small_sota_best.pt
Epoch  5/10 [669.2s] | Train Loss: 0.6315 | EMA Val Acc: 97.88%
  --> New Peak EM

## 5. Test Set Benchmarking

In [ ]:
# Cell 7: Final Test Set Evaluation
if os.path.exists(best_ckpt_path):
    model_best = build_model_sota(model_name="convnext_small", num_classes=10, device=device)
    model_best.load_state_dict(torch.load(best_ckpt_path, map_location=device))
    test_loss, test_acc = evaluate_exp08(model_best, test_loader, device)
    print("="*60)
    print(f"EXP-08 CONVNEXT-SMALL FINAL TEST ACCURACY: 🏆 {test_acc:.2f}%")
    print("="*60)
else:
    print(f"[Test] Run Cell 6 first to generate checkpoint.")


EXP-08 CONVNEXT-SMALL FINAL TEST ACCURACY: 🏆 98.62%


: 